# 02. CNN Baseline (ResNet18 Transfer Learning)

ResNet18 ImageNet pretrained → 4-class 자세 분류

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import torch
import json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.dataset import PostureImageDataset
from src.models import build_resnet18, build_efficientnet_b0
from src import CLASSES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# 데이터로더 준비
SPLITS_CSV = 'data/splits/splits.csv'
IMG_ROOT = 'data/raw'

train_ds = PostureImageDataset(SPLITS_CSV, IMG_ROOT, split='train')
val_ds = PostureImageDataset(SPLITS_CSV, IMG_ROOT, split='val')
test_ds = PostureImageDataset(SPLITS_CSV, IMG_ROOT, split='test')

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
# 샘플 배치 시각화
import torchvision
import numpy as np

imgs, labels = next(iter(train_loader))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
imgs_denorm = (imgs[:8] * std + mean).clamp(0, 1)

grid = torchvision.utils.make_grid(imgs_denorm, nrow=4)
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1,2,0).numpy())
plt.title('Train 배치 샘플 | ' + ' | '.join([CLASSES[l] for l in labels[:8].tolist()]))
plt.axis('off')
plt.tight_layout()
plt.savefig('results/figures/train_batch_sample.png', dpi=150)
plt.show()

In [ ]:
# 학습 실행 (스크립트 방식 권장, 여기서는 직접 실행)
# CLI 방식: python -m src.train_cnn --backbone resnet18 --epochs 30

import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'src.train_cnn',
     '--backbone', 'resnet18',
     '--epochs', '30',
     '--batch_size', '32',
     '--lr', '1e-4'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('[STDERR]', result.stderr[-1000:])

In [ ]:
# 학습 곡선 로드 및 시각화
with open('results/metrics/cnn_resnet18_history.json') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'], label='Val')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()